In [1]:
# Cell 1
import os
import time
import json
import torch
from tqdm.notebook import tqdm
from dotenv import load_dotenv
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    TrainingArguments, 
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType

load_dotenv()

# Setup device
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

# Define paths
PROCESSED_DATA_DIR = "../data/processed"
MODELS_DIR = "../models"
os.makedirs(MODELS_DIR, exist_ok=True)

CHUNKS_FILE = os.path.join(PROCESSED_DATA_DIR, "chunks.json")
ADAPTER_SAVE_PATH = os.path.join(MODELS_DIR, "tisd-tinyllama-lora")

Using device: mps
PyTorch version: 2.11.0


In [2]:
# Cell 2
print("Loading chunks to create a training dataset...")
start_time = time.time()

with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

# We will take 50 chunks to make our training fast for this project demo
train_chunks = all_chunks[:50] 

qa_dataset = []
for chunk in tqdm(train_chunks, desc="Generating Q&A Pairs"):
    context = chunk["chunk_text"]
    subject = chunk["subject"]
    
    # Create a generic question
    question = f"Can you explain what this text about {subject} teaches us?"
    
    # Create a friendly, teacher-like answer grounded ONLY in the text
    answer = f"Hello there! I'd love to help. Based on the text, we learn that: {context} Keep up the great reading!"
    
    qa_dataset.append({
        "context": context,
        "question": question,
        "answer": answer
    })

elapsed_time = time.time() - start_time
print(f"Created {len(qa_dataset)} training examples in {elapsed_time:.2f} seconds.")

# Convert to HuggingFace Dataset format
hf_dataset = Dataset.from_list(qa_dataset)
print(hf_dataset[0]) # Print sample

Loading chunks to create a training dataset...


Generating Q&A Pairs:   0%|          | 0/50 [00:00<?, ?it/s]

Created 50 training examples in 0.03 seconds.
{'context': 'Take a bus Or take a train, Take a boat Or take a plane, Take a taxi, Take a car, Maybe near Or maybe far, Sight words take | is | the | or | two | may be | Unit 3 Going places Chapter 1 Come Back Soon Let us recite Chapter 3.indd 37Chapter 3.indd 37 19-05-2023 10:44:3419-05-2023 10:44:34 Reprint 2026-27', 'question': 'Can you explain what this text about English teaches us?', 'answer': "Hello there! I'd love to help. Based on the text, we learn that: Take a bus Or take a train, Take a boat Or take a plane, Take a taxi, Take a car, Maybe near Or maybe far, Sight words take | is | the | or | two | may be | Unit 3 Going places Chapter 1 Come Back Soon Let us recite Chapter 3.indd 37Chapter 3.indd 37 19-05-2023 10:44:3419-05-2023 10:44:34 Reprint 2026-27 Keep up the great reading!"}


In [3]:
# Cell 3
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Loading Tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# TinyLlama doesn't have a pad token by default, so we assign one
tokenizer.pad_token = tokenizer.eos_token 

def format_and_tokenize(example):
    """Formats the data into TinyLlama's expected Prompt structure."""
    system_prompt = "You are a friendly teacher for grade 1 to 4 students. Answer the question using only the provided context. Use simple words."
    
    # Construct the prompt
    full_prompt = f"<|system|>\n{system_prompt}</s>\n<|user|>\nContext: {example['context']}\nQuestion: {example['question']}</s>\n<|assistant|>\n{example['answer']}</s>"
    
    # Tokenize
    tokens = tokenizer(full_prompt, truncation=True, max_length=256, padding="max_length")
    
    # For training Causal LMs, the labels are the same as the input_ids
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

print("Tokenizing dataset...")
start_time = time.time()

tokenized_dataset = hf_dataset.map(format_and_tokenize, remove_columns=["context", "question", "answer"])

elapsed_time = time.time() - start_time
print(f"Dataset tokenized in {elapsed_time:.2f} seconds.")

Loading Tokenizer: TinyLlama/TinyLlama-1.1B-Chat-v1.0


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Tokenizing dataset...


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Dataset tokenized in 0.22 seconds.


In [4]:
# Cell 4
print(f"Loading base model {MODEL_ID} to {device}...")
start_time = time.time()

# Load model in bfloat16 (Apple Silicon supports this natively)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map={"": device} # Forces model onto MPS
)

# Define LoRA Configuration
# r=8 means we are creating a very small, fast-to-train adapter
lora_config = LoraConfig(
    r=8, 
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"], # Target attention mechanisms
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Wrap the base model with the LoRA adapter
model = get_peft_model(model, lora_config)

elapsed_time = time.time() - start_time
print(f"Model and LoRA loaded in {elapsed_time:.2f} seconds.")
model.print_trainable_parameters() # This will show you're only training ~1% of weights!

Loading base model TinyLlama/TinyLlama-1.1B-Chat-v1.0 to mps...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model and LoRA loaded in 419.42 seconds.
trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [8]:
# Cell 5
print("Setting up training arguments...")

training_args = TrainingArguments(
    output_dir=os.path.join(MODELS_DIR, "checkpoints"),
    per_device_train_batch_size=2,      # Small batch size to ensure no Memory out-of-bounds
    gradient_accumulation_steps=4,      # Accumulate gradients to simulate batch_size=8
    learning_rate=2e-4,
    num_train_epochs=3,                 # 3 passes over our mini-dataset
    logging_steps=5,
    save_strategy="no",                 # Don't save intermediate steps to save disk space
    optim="adamw_torch",                # Standard optimizer compatible with MPS
    report_to="none"                    # Disables wandb logging
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Starting training loop...")
start_time = time.time()

# ACTUALLY TRAIN THE MODEL!
trainer.train()

elapsed_time = time.time() - start_time
print(f"Training completed beautifully in {elapsed_time:.2f} seconds!")

Setting up training arguments...
Starting training loop...


/opt/homebrew/Caskroom/miniforge/base/envs/tisd/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
5,2.729639
10,2.735777
15,2.474784
20,2.388994


Training completed beautifully in 90.59 seconds!


In [9]:
# Cell 6
print(f"Saving fine-tuned LoRA adapter to {ADAPTER_SAVE_PATH}...")
model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

print("✅ Model saved successfully! You are ready for Notebook 4.")

Saving fine-tuned LoRA adapter to ../models/tisd-tinyllama-lora...
✅ Model saved successfully! You are ready for Notebook 4.
